In [1]:
import re
import pdfplumber
import pandas as pd

# Table 18 of the CBK Bank Supervision Annual Report 2025 sits on PDF page 70
with pdfplumber.open("data/ANNUAL_REPORT_2025.pdf") as pdf:
    text = pdf.pages[69].extract_text()

num = r"[\d,]+(?:\.\d+)?"
full = re.compile(rf"^(.*?)\s+({num})\s+({num})\s+({num})\s+({num})\s+({num})\s+({num})$")
nums_only = re.compile(rf"^({num})\s+({num})\s+({num})\s+({num})\s+({num})\s+({num})$")

block = text[text.find("Table 18"):text.find("Source: CBK")].split("\n")

rows = []
for i, line in enumerate(block):
    m = full.match(line.strip())
    name = None
    if m:
        name = m.group(1)
        vals = m.groups()[1:]
    else:
        m = nums_only.match(line.strip())
        if m:  # sector name wraps over two lines
            name = block[i-1].strip() + " " + block[i+1].strip()
            vals = m.groups()
    if name:
        rows.append([name] + [float(v.replace(",", "")) for v in vals])

df = pd.DataFrame(rows, columns=["sector", "loan_accounts", "pct_accounts",
                                 "gross_loans_ksh_m", "pct_loans",
                                 "gross_npl_ksh_m", "pct_npl"])

# Check: the 11 sectors should add up to the published totals
d = df[df["sector"] != "Total"]
t = df[df["sector"] == "Total"].iloc[0]
print("Loans:", round(d["gross_loans_ksh_m"].sum(), 1), "vs published", t["gross_loans_ksh_m"])
print("NPLs: ", round(d["gross_npl_ksh_m"].sum(), 1), "vs published", t["gross_npl_ksh_m"])

# Default rate = a sector's bad loans as a share of its own loans
df["npl_ratio_pct"] = (df["gross_npl_ksh_m"] / df["gross_loans_ksh_m"] * 100).round(1)

sectors = df[df["sector"] != "Total"].sort_values("npl_ratio_pct", ascending=False)
print(sectors[["sector", "pct_loans", "pct_npl", "npl_ratio_pct"]].to_string(index=False))

df.to_csv("data/sector_npls_dec2025.csv", index=False)
print("Saved")

Loans: 4347164.1 vs published 4347164.1
NPLs:  696931.1 vs published 696931.0
                        sector  pct_loans  pct_npl  npl_ratio_pct
     Building and Construction        4.0      7.4           29.5
Tourism, Restaurant and Hotels        2.3      3.3           22.9
                   Real Estate       11.8     16.3           22.3
                         Trade       19.7     24.5           19.9
          Mining and Quarrying        0.9      1.1           19.7
                   Agriculture        4.6      5.5           19.3
                 Manufacturing       13.5     15.4           18.4
   Transport and Communication        7.4      7.6           16.6
        Personal and Household       27.1     16.4            9.7
              Energy and Water        3.3      1.4            6.8
            Financial Services        5.6      1.1            3.3
Saved
